---
# Linear Regression 
---

In [ ]:
%load_ext autoreload
%autoreload 2

import model
import torch
import plotly.express as px
import pandas as pd
import tests

---

Linear Regression tries to predict data which lives on a continuous domain like the $\mathbb{R}^n$. For this notebook, we assume the output to be a single scalar, but this can be expanded easily by combining the model parameters. We will train this linear regression model with gradient descent.

The prediction function of a linear regression model is
$$
\hat{y} = X w + b \cdot \mathbb{1}_N
$$
where the data matrix contains $N$ data points of dimension $D$ as rows $X \in \mathbb{R}^{N \times D}$, the weight is a vector $w \in \mathbb{R}^D$ and the bias a scalar $b \in \mathbb{R}$.

Which shape has the output $y$?

---
## Mean Squared Error Loss (MSE)
---

To train a model with gradient descent, we need a scalar loss function. For this regression task we will use the MSE.
$$
L(\hat{y}, y) = \frac{1}{N} \sum_i (\hat{y}_i - y_i)^2
$$

---

## **Task:**

Implement `mse` in `model.py`.

**Hints:**

- To square the elements of a tensor, you can use `tensor ** 2` or `torch.pow`.
- Use `torch.sum` to calculate sums.
- Use `torch.mean` to calculate means.

In [2]:
tests.test_mse(model.mse)

MSE is correct



---

## MSE Gradient

---

Before we can calculate the gradients for our model parameters, we first need the gradient of the loss function with respect to the model output.

## **Task:**
Calculate the gradient of the MSE on paper. Then implement `mse_gradients` in `model.py`, which returns the gradient vector for the input `y_pred`.

**Hints:**

- To calculate the gradient, you need the chain rule.
- While the MSE is symmetric in its arguments, the gradient function is not. It is antisymetric w.r.t the parameter of differentiation $\nabla_{\hat{y}} L(\hat{y}, y) = - \nabla_{y} L(\hat{y}, y)$.
- To get the number of elements $N$ of a tensor, you can use `tensor.shape` or `len(tensor)` for the first dimension.

In [3]:
tests.test_mse_grad(model.mse_gradients)

The gradient is correct


---
## Linear Model
---

### Prediction

## **Task:**

Implement `linear_predict` in `model.py`, which returns the model prediction $\hat{y}$.

**Hints:**

- If you're stuck, try to implement the function for a single datapoint first.
- For batch data, the bias can be broadcasted automatically without any explicit function. But try to broadcast yourself!

In [4]:
tests.test_linear_predict(model.linear_predict)

Single data: The prediction is correct
Batch data: The prediction is correct



---

## Linear Model - Gradient

---

## **Task:**

Calculate the gradient for **all** model parameters on paper. Then implement `linear_gradients` in `model.py`.

Hints:

- Use the chain rule to reuse the gradient of the loss function.
- If you're stuck, try to implement the function for a single datapoint first.
- The gradient of a tensor must have the same shape as the tensor.
- As the bias is used in multiple predictions (broascasted with $\mathbb{1}_N$), the total gradient is sum of the gradients of b for single data points.


In [6]:
tests.test_linear_gradients(model.linear_gradients)

Single data: The gradient for the weight is correct
Single data: The gradient for the bias is correct
Batch data: The gradient for the weight is correct
Batch data: The gradient for the bias is correct


---

## Gradient Descent

---

Gradient descent updates all model parameters $\theta$ of a model $f_\theta$ in the direction of the negative gradient, scaled by a learning rate $\eta$.

$$
\theta_{t+1} = \theta_t - \eta * \nabla_\theta \mathcal{L}( f_\theta(X), y)
$$

## **Task:**

Implement `fit` in `model.py` that computes stochastic gradient descent.

**Hints:**

- For gradient descent, the gradient needs to be **subtracted**.
- The model parameters are both the _weights_ and the _bias_.

In [7]:
tests.test_fit(model.fit)

The new weight is correct.
The new bias is correct


---
## Dataset: WHO Life Expectancy
---
We now train a model on a real dataset, which measures the life expectancy of different countries, based on many different parameters.

In [13]:
data = torch.load("who_data.pt", weights_only=False)
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']

dataset = torch.utils.data.TensorDataset(X_train, y_train)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

---

## Parameter Search

---

## **Task:**

Find a good learning rate to fit the model (We get ~13.2 test error). 

**Hints:**

- If $\eta$ is too low, the learning will be slow.
- If $\eta$ is too large, training is unstable and a good weight can't be reached.
- If the plot is empty, there are probably NaNs (training fails due to bad learning rate).

In [15]:
#############################################
# TODO
learning_rate = 0.06
#############################################
epochs = 50

w, b, error_history = model.fit(loader, epochs=epochs, learning_rate=learning_rate)
fig = px.line(x=range(epochs), y=error_history, log_y=True, title="Error during Training").show()
test_err = model.mse(y_test, model.linear_predict(X_test, w, b))
print(f"The test error is {test_err:.2f}")

The test error is 13.14


---

## Interpret the Model

---

Look at the weight for each parameter and think about what they mean and if all of them are reasonable. 

What does a positive/negative value mean?

In [16]:
w_df = pd.DataFrame(index=data['names'], data=w.numpy())
display(w_df)


,0
Year,-0.122085
Adult Mortality,-2.072492
infant deaths,2.389452
Alcohol,-0.565077
percentage expenditure,0.697842
Hepatitis B,-0.195668
Measles,0.080667
BMI,0.600770
under-five deaths,-2.767013
Polio,0.228275
